In [0]:
use catalog employee;
use schema bronze;

In [0]:
CREATE OR REPLACE TABLE employee.silver.workforce_silver
USING DELTA
AS
SELECT
    e.Employee_ID,
    e.full_name,
    e.Age,
    e.Gender,
    e.Department,
    e.Job_Title,
    e.Experience_Years,
    e.Education_Level,
    e.Location,

    s.base_salary,
    s.Bonus,
    s.Allowances,

    (s.base_salary + COALESCE(s.Bonus,0) + COALESCE(s.Allowances,0)) AS total_salary

FROM employee.bronze.employees_raw e
LEFT JOIN employee.bronze.salaries_raw s
    ON e.Employee_ID = s.Employee_ID;

In [0]:
select * from employee.silver.workforce_silver

In [0]:
select * from employee.silver.workforce_clean

In [0]:
CREATE OR REPLACE TABLE employee.silver.workforce_clean
USING DELTA
AS
SELECT DISTINCT
    Employee_ID,
    TRIM(full_name) AS full_name,
    Age,
    TRIM(Gender) AS Gender,
    TRIM(Department) AS Department,
    TRIM(Job_Title) AS Job_Title,
    Experience_Years,
    TRIM(Education_Level) AS Education_Level,
    TRIM(Location) AS Location
FROM employee.silver.workforce_silver
WHERE Employee_ID IS NOT NULL;

In [0]:
CREATE OR REPLACE TABLE employee.silver.attendance_silver
USING DELTA
AS
SELECT
    e.Employee_ID,
    e.full_name,
    e.Department,

    COUNT(a.Work_Date) AS total_days,
    SUM(a.Hours_Worked) AS total_hours,
    a.status,
   

    SUM(l.Leave_Days) AS leave_days

FROM employee.bronze.employees_raw e
LEFT JOIN employee.bronze.attendance_raw a
    ON e.Employee_ID = a.Employee_ID
LEFT JOIN employee.bronze.leaves_raw l
    ON e.Employee_ID = l.Employee_ID

GROUP BY
    e.Employee_ID, e.full_name, e.Department, a.status;

In [0]:
select * from employee.silver.attendance_silver

In [0]:
select * from employee.silver.attendance_clean

In [0]:
CREATE OR REPLACE TABLE employee.silver.attendance_clean
USING DELTA
AS
SELECT DISTINCT
    Employee_ID,

    TRIM(full_name) AS full_name,
    TRIM(Department) AS Department,

    COALESCE(total_days,0) AS total_days,
    COALESCE(total_hours,0) AS total_hours,

    TRIM(status) AS status,

    COALESCE(leave_days,0) AS leave_days

FROM employee.silver.attendance_silver
WHERE Employee_ID IS NOT NULL;

In [0]:
CREATE OR REPLACE TABLE employee.silver.compensation_silver
USING DELTA
AS
SELECT
    e.Employee_ID,
    e.full_name,
    e.Department,
    e.Experience_Years,

    s.base_salary,
    s.Bonus,
    s.Allowances,
    s.Tax_Percent,

    SUM(a.hours_worked) AS total_overtime,

    (s.base_salary 
        + COALESCE(s.Bonus,0) 
        + COALESCE(s.Allowances,0)
        + SUM(a.hours_worked)*20) AS gross_pay

FROM employee.bronze.employees_raw e
LEFT JOIN employee.bronze.salaries_raw s
    ON e.Employee_ID = s.Employee_ID
LEFT JOIN employee.bronze.attendance_raw a
    ON e.Employee_ID = a.Employee_ID

GROUP BY
    e.Employee_ID,
    e.full_name,
    e.Department,
    e.Experience_Years,
    s.base_salary,
    s.Bonus,
    s.Allowances,
    s.Tax_Percent;

In [0]:
select * from employee.silver.compensation_silver

In [0]:
select * from employee.silver.compensation_clean

In [0]:
CREATE OR REPLACE TABLE employee.silver.compensation_clean
USING DELTA
AS
SELECT DISTINCT
    Employee_ID,
    TRIM(full_name) AS full_name,
    TRIM(Department) AS Department,
    Experience_Years,
    COALESCE(base_salary,0) AS base_salary,
    COALESCE(Bonus,0) AS Bonus,
    COALESCE(Allowances,0) AS Allowances,
    COALESCE(Tax_Percent,0) AS Tax_Percent,
    COALESCE(total_overtime,0) AS total_overtime,
    COALESCE(gross_pay,0) AS gross_pay
FROM employee.silver.compensation_silver
WHERE Employee_ID IS NOT NULL;

In [0]:
CREATE OR REPLACE TABLE employee.silver.performance_silver
USING DELTA
AS
SELECT
    e.Employee_ID,
    e.full_name,
    e.Department,
    e.Job_Title,

    p.Rating,
    p.Grade,

    COUNT(t.Course_Name) AS courses_taken,
    AVG(t.Score) AS avg_training_score,

    CASE
        WHEN p.Rating >= 4 THEN 'High'
        WHEN p.Rating >= 3 THEN 'Medium'
        ELSE 'Low'
    END AS performance_band

FROM employee.bronze.employees_raw e
LEFT JOIN employee.bronze.performance_raw p
    ON e.Employee_ID = p.Employee_ID
LEFT JOIN employee.bronze.training_raw t
    ON e.Employee_ID = t.Employee_ID

GROUP BY
    e.Employee_ID, e.full_name, e.Department, e.Job_Title,
    p.Rating, p.Grade;

In [0]:
select * from employee.silver.performance_silver

In [0]:
select * from employee.silver.performance_clean

In [0]:
CREATE OR REPLACE TABLE employee.silver.performance_clean
USING DELTA
AS
SELECT DISTINCT
    Employee_ID,
    
    TRIM(full_name) AS full_name,
    TRIM(Department) AS Department,
    TRIM(Job_Title) AS Job_Title,

    COALESCE(Rating,0) AS Rating,
    TRIM(Grade) AS Grade,

    COALESCE(courses_taken,0) AS courses_taken,
    COALESCE(avg_training_score,0) AS avg_training_score,

    TRIM(performance_band) AS performance_band

FROM employee.silver.performance_silver
WHERE Employee_ID IS NOT NULL;